# Session 0 · Before the Machine: Structure Without a Subject
*Cultural Machines: An Introduction*
Based on Leif Weatherby, *Language Machines: Cultural AI and the End of Remainder Humanism* (University of Minnesota Press, 2025).

**This is the lead notebook for the course.** It sets up the question every later session returns to: *where does meaning come from, if not from a mind?*

**In this session you will:** compare two stories about modern thought (Michel Foucault's and Sarah Pourciau's), play a game whose pieces don't matter, strip sounds down to their differences, build numbers out of nothing, watch repetition create order, and turn a map of words upside down without changing what it says.

**Time:** about 75 minutes. No GPU needed: the standard runtime is fine and every box runs instantly.

### How to use this notebook
- This is a **Google Colab notebook**: a page that mixes reading with small pieces of code you can run.
- To run a grey code box, click it and press **Shift + Enter** (or click the ▶ button on its left).
- **Run the boxes in order, top to bottom.** If something breaks, go to *Runtime → Restart session* and start again from the top.
- You never have to *write* code. Where you see text inside quotation marks, like `"this"`, you can change the words and run the box again. That is the whole skill.
- Boxes marked **Setup** load the machinery. You can open them if you are curious, but you do not need to read them.

In [ ]:
#@title Setup: load the tools for this session (instant, no GPU needed)
import re, random, math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt

# ---------- Part 1: sorting statements ----------
KEY = {"A": "human intention", "B": "mere statistics", "C": "a self-organising system"}
def tally(answers):
    counts = {k: 0 for k in KEY}
    for statement, letter in answers.items():
        letter = letter.strip().upper()
        if letter not in KEY:
            print(f"Skipped (use A, B or C): {statement}")
            continue
        counts[letter] += 1
        print(f"{letter}  {statement}")
    print()
    for k, v in counts.items():
        print(f"{KEY[k]:<26} {'■' * v} {v}")

# ---------- Part 2: Saussure's chess ----------
LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
GAME = [5, 2, 1, 9, 7, 3, 4]   # squares numbered 1-9, left to right, top to bottom
def play(piece_1, piece_2, moves=GAME):
    board = ["·"] * 9
    for turn, square in enumerate(moves):
        board[square - 1] = piece_1 if turn % 2 == 0 else piece_2
    for r in range(3):
        print("   " + "   ".join(board[r*3:r*3+3]))
    for a, b, c in LINES:
        if board[a] != "·" and board[a] == board[b] == board[c]:
            who = "first" if board[a] == piece_1 else "second"
            print(f"\nThe program declares a winner: {board[a]} (the {who} player), squares {a+1}, {b+1}, {c+1}.")
            return
    print("\nNo winner.")

# ---------- Part 3: Jakobson's features ----------
FEATURES = ["voiced", "nasal", "lips", "hissing"]
SOUNDS = {  # 1 = has the feature (marked), 0 = lacks it (the zero, unmarked)
    "p": (0,0,1,0), "b": (1,0,1,0), "m": (1,1,1,0), "f": (0,0,1,1), "v": (1,0,1,1),
    "t": (0,0,0,0), "d": (1,0,0,0), "n": (1,1,0,0), "s": (0,0,0,1), "z": (1,0,0,1),
}
def sound_table(ignore=()):
    keep = [i for i, f in enumerate(FEATURES) if f not in ignore]
    print("sound  " + "  ".join(f"{FEATURES[i]:>7}" for i in keep))
    for s, v in SOUNDS.items():
        print(f"  {s}    " + "  ".join(f"{('+' if v[i] else '0'):>7}" for i in keep))
    groups = {}
    for s, v in SOUNDS.items():
        groups.setdefault(tuple(v[i] for i in keep), []).append(s)
    merged = [g for g in groups.values() if len(g) > 1]
    print()
    if merged:
        for g in merged:
            print("Can no longer be told apart: " + " = ".join(g))
    else:
        print("Every sound is still distinct. Each one is nothing but its pattern of + and 0.")

# ---------- Part 4a: Dedekind's cut ----------
def dedekind_cut(target=2, max_denominator=12):
    if not (0 < target <= 9):
        print("Please choose a target between 1 and 9."); return
    fracs = sorted({Fraction(n, d) for d in range(1, max_denominator + 1) for n in range(0, 3 * d + 1)})
    A = [q for q in fracs if q * q < target]
    B = [q for q in fracs if q * q > target]
    exact = [q for q in fracs if q * q == target]
    print(f"Fractions checked: {len(fracs)} (bottoms up to {max_denominator})")
    print(f"Biggest fraction in the LOWER group A:  {str(A[-1]):>7}  = {float(A[-1]):.6f}")
    print(f"Smallest fraction in the UPPER group B: {str(B[0]):>7}  = {float(B[0]):.6f}")
    if exact:
        print(f"\nA fraction sits exactly at the split: {exact[0]}. No new number is needed here.")
    else:
        print(f"\nNo fraction sits between A and B, however far you look.")
        print(f"Dedekind's move: the gap itself IS the number we write √{target}.")
    centre = math.sqrt(target)
    fig, ax = plt.subplots(figsize=(10, 1.8))
    lo, hi = centre - 0.35, centre + 0.35
    a = [float(q) for q in A if lo < q < hi]; b = [float(q) for q in B if lo < q < hi]
    ax.scatter(a, [0]*len(a), label="A: squared, less than target", s=25)
    ax.scatter(b, [0]*len(b), label="B: squared, more than target", s=25, marker="s")
    if exact:
        ax.scatter([float(exact[0])], [0], s=90, marker="*", label="a fraction at the split")
    else:
        ax.axvline(centre, linestyle="--", linewidth=1)
        ax.text(centre, 0.3, "the cut", ha="center")
    ax.set_yticks([]); ax.set_xlim(lo, hi); ax.set_ylim(-0.5, 0.6)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -1.1), ncol=3, frameon=False)
    ax.set_title(f"Fractions near √{target}")
    plt.show()

# ---------- Part 4b: numbers from nothing ----------
def show_set(s):
    if not s:
        return "∅"
    return "{" + ", ".join(show_set(x) for x in sorted(s, key=len)) + "}"
def numbers_from_nothing(up_to=4):
    numbers = [frozenset()]
    for n in range(1, up_to + 1):
        numbers.append(frozenset(numbers))
    for i, s in enumerate(numbers):
        text = show_set(s)
        if len(text) > 100:
            text = text[:97] + "..."
        print(f"{i} = {text}")
    print(f"\nThe only raw material used anywhere above is ∅, the empty collection.")
    print(f"'{up_to}' is written with {len(show_set(numbers[-1]))} characters, and every one of its parts is built from nothing.")
    return numbers

# ---------- Part 4c: rhyme makes order ----------
def plain_collection(a, b):
    return frozenset([a, b])
def ordered_pair(a, b):
    return frozenset([frozenset([a]), frozenset([a, b])])
def compare(a, b):
    print(f"Plain collection:  {{{a}, {b}}} is the same as {{{b}, {a}}}?  {plain_collection(a, b) == plain_collection(b, a)}")
    print(f"Kuratowski pair:   ({a}, {b}) is the same as ({b}, {a})?  {ordered_pair(a, b) == ordered_pair(b, a)}")
    print(f"\n({a}, {b}) is built as {{ {{{a}}}, {{{a}, {b}}} }}")
    print(f"'{a}' appears twice, '{b}' once. That repetition is the only thing that makes '{a}' come first.")
def rhyme_scheme(poem, letters=2):
    lines = [l.strip() for l in poem.strip().split("\n") if l.strip()]
    labels, scheme = {}, []
    for line in lines:
        words = re.findall(r"[a-z']+", line.lower())
        if not words:
            continue
        ending = words[-1][-letters:]
        if ending not in labels:
            labels[ending] = chr(ord("A") + len(labels))
        scheme.append(labels[ending])
        print(f"{labels[ending]}   {line}")
    print("\nRhyme scheme: " + "".join(scheme))

# ---------- Part 4e: rule, not list ----------
PEOPLE = ["the curator", "the painter", "the drummer", "the critic", "the poet"]
ACTIONS = ["admired", "ignored", "invited", "remembered", "photographed"]
ENDINGS = ["left early", "stayed until midnight", "wrote the review", "sold the painting"]
def noun_phrase(depth):
    phrase = random.choice(PEOPLE)
    if depth > 0:
        phrase += " who " + random.choice(ACTIONS) + " " + noun_phrase(depth - 1)
    return phrase
def sentences(depth=2, how_many=5, seed=1):
    random.seed(seed)
    for _ in range(how_many):
        s = noun_phrase(depth) + " " + random.choice(ENDINGS) + "."
        print("• " + s[0].upper() + s[1:])
    print("\nHow many different sentences one short rule can make:")
    for d in range(0, 6):
        count = len(PEOPLE) * (len(ACTIONS) * len(PEOPLE)) ** d * len(ENDINGS)
        print(f"  with {d} 'who' clauses: {count:,}")
def sets_in_stages(stages=7):
    size = 0
    for k in range(min(stages, 7)):
        if k < 6:
            print(f"Stage {k}: {size:,} sets")
            size = 2 ** size
        else:
            digits = int(65536 * math.log10(2)) + 1
            print(f"Stage {k}: a number of sets with {digits:,} digits")
    print("Stage 7 and beyond: too large for any computer to write down. There is no last stage.")

# ---------- Part 5: Benacerraf in the machine ----------
WORD_MAP = {
    "painting": (1.0, 2.0), "drawing": (1.4, 2.5), "sculpture": (0.5, 2.4),
    "song": (4.0, 0.9), "drum": (4.5, 0.4), "guitar": (3.6, 0.3),
    "actor": (2.4, -1.8), "script": (3.0, -2.2), "stage": (2.1, -2.5),
}
def nearest(points):
    names = list(points)
    out = {}
    for w in names:
        others = [(np.linalg.norm(np.subtract(points[w], points[o])), o) for o in names if o != w]
        out[w] = min(others)[1]
    return out
def rotate_map(angle=90, mirror=False):
    t = math.radians(angle)
    R = np.array([[math.cos(t), -math.sin(t)], [math.sin(t), math.cos(t)]])
    moved = {}
    for w, p in WORD_MAP.items():
        q = R @ np.array(p)
        if mirror:
            q[0] = -q[0]
        moved[w] = tuple(q)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, pts, title in [(axes[0], WORD_MAP, "Original map"), (axes[1], moved, f"Turned {angle}°" + (" and mirrored" if mirror else ""))]:
        xs, ys = zip(*pts.values())
        ax.scatter(xs, ys, s=30)
        for w, (x, y) in pts.items():
            ax.annotate(w, (x, y), textcoords="offset points", xytext=(5, 5))
        ax.set_title(title); ax.set_aspect("equal"); ax.margins(0.25)
        ax.set_xticks([]); ax.set_yticks([])
    plt.show()
    before, after = nearest(WORD_MAP), nearest(moved)
    print("Closest neighbour of each word:")
    for w in WORD_MAP:
        print(f"  {w:<10} before: {before[w]:<10} after: {after[w]:<10} {'same' if before[w] == after[w] else 'CHANGED'}")
    print("\nEvery coordinate changed. Every relationship survived.")

print("Ready.")

## The big idea

Public debate about AI tends to offer two answers to the question "where does meaning come from?"

- **Human intention.** Words mean something because a person meant something by them. On this view, machines only ever imitate meaning.
- **Mere statistics.** Language models just count which words follow which. On this view, machine meaning is an illusion produced by frequency.

This session introduces a **third answer**: meaning can come from a real, self-organising *system*, one that is neither a human subject nor a hidden inner spirit. That third answer comes from structuralist linguistics, and it turns out to have a twin in modern mathematics.

Our guide is Sarah Pourciau's *The Writing of Spirit* (2017), a history of how linguistics became a science. It is the backstory to Leif Weatherby's *Language Machines*, which the rest of the course follows. Weatherby argues that to understand language models we need to go back to structuralism; Pourciau shows what structuralism actually was.

## Part 1 · Two stories about modern thought

### Foucault's story: the rise and fall of "man"

In *The Order of Things* (1966), the French philosopher Michel Foucault described two great breaks in Western knowledge.

**Around 1800**, knowledge stopped being a simple mirror of the world. The philosopher Immanuel Kant made the *knower* the central question, and the human subject moved to the centre. New "sciences of man" appeared: biology, economics, and the study of language (philology). The German scholars who founded modern linguistics, such as Franz Bopp and Jacob Grimm, belong on this reading to a human-centred age.

**In the twentieth century**, structuralist linguistics (Ferdinand de Saussure, Roman Jakobson) described systems that produce meaning with no subject at the centre. Foucault suggested this might mean the end of "man", who could vanish *like a face drawn in sand at the edge of the sea*.

The result is the classic story: the nineteenth century was about people making meaning through history; the twentieth discovered structure without a subject.

### Pourciau's correction

Pourciau agrees the shift around 1800 comes after Kant. But she argues it was **against** Kant, and that it followed a different philosopher: Friedrich Schelling.

- **Kant** said order is something the mind brings to experience. We cannot know whether nature itself is organised.
- **The scientists of the period** refused that division between mind and world. They believed order was *really out there*, part of the cosmos itself. Their problem was explaining that order without a God shaping matter from outside.

Their answer: order **develops from within**, over time. Schelling called this principle the *world soul*. Biologists called it the *formative principle*. Linguists called it the *Sprachgeist*, or "language spirit". None of these is a human subject. Each is an impersonal, self-organising force, and people are participants in it, not its authors.

Pourciau also spots a telling detail. Foucault builds his account of the new biology around the Frenchman Georges Cuvier and ignores earlier German biologists (Blumenbach and Kielmeyer) who already treated living systems as types unfolding in time. Change whose science sits at the centre, and the picture changes.

### So what did structuralism actually break from?

1. **Structuralism did not discover subject-free structure.** The nineteenth century already had it. So "removing the human" cannot be what made structuralism new.
2. **It broke from a system driven by an inner spirit**: a living force with tendencies, direction and a goal. Saussure's target was spirit, depth and inner life, not humanism.
3. **It kept the idea that systems are real and self-organising, and removed the content doing the organising.** The unifying role passed to an empty term that can be written down. Pourciau's name for this is "the writing of spirit".

**An analogy.** For Foucault, the nineteenth century is a play with a human playwright, and structuralism reveals there is no playwright, only the script's internal logic. For Pourciau, the nineteenth century had *already* said there is no human playwright: the play writes itself, driven by a living spirit inside it. Structuralism's move was to say there is no inner spirit either. The script's unity comes from its structure of differences.

### Activity 1 · Where does meaning come from? (10 minutes)

Beside each statement, put **"A"** (human intention), **"B"** (mere statistics) or **"C"** (a self-organising system). There are no right answers. Run the box to see your spread, then compare with a neighbour.

In [ ]:
my_answers = {
    "A poem means what its author intended.": "A",
    "A chatbot's reply is just the most likely next words.": "B",
    "A dictionary defines words only by other words.": "C",
    "The meaning of a proverb belongs to no one person.": "C",
    "A song can mean something its writer never noticed.": "A",
    "A translation app understands nothing.": "B",
    "A child learns a word's meaning by hearing it used.": "B",
}
tally(my_answers)

**Talk it over.** Which statements were hardest to place? Did anything feel like it belonged in all three columns? Keep your answers: we return to them at the end.

## Part 2 · Role, not material

Saussure's most famous claim is that language is a system of **pure differences**: a sign means something only by contrast with other signs, not because of what it is made of. He explained it with chess. If you lose a knight, you can replace it with a button, as long as it plays the knight's role.

He called this the **arbitrariness of the sign**. Pourciau shows it was aimed at the German idea of language spirit: meaning does not come from some inner essence of the material.

### Activity 2 · Change the pieces (5 minutes)

Below is a finished game of noughts and crosses. Run it. Then replace `"X"` and `"O"` with anything you like: `"🥁"` and `"🎨"`, `"cat"` and `"dog"`, your initials. Does the game change?

**Then try this:** make both pieces the same, for example `"O"` and `"O"`. Read what the program says now.

In [ ]:
play("X", "O")

With two identical pieces, the program sees a winner that never happened. Once the pieces can't be told apart, the game collapses. What mattered was never the material, only the **difference**. That is Saussure's point about language.

## Part 3 · The zero

Roman Jakobson took Saussure's idea down to the level of sounds. He described each speech sound as a bundle of yes/no features: is it voiced (do the vocal cords buzz)? nasal (does air go through the nose)? made with the lips? hissing?

A sound that **has** a feature is *marked* for it. A sound that **lacks** it is *unmarked*, which Jakobson called the **zero**. The zero is a real position in the system, but its only content is absence.

Pourciau calls the zero **a placeholder for the subtracted soul of structure**. The old language spirit has been emptied out, and the emptiness itself is written into the system as one of its terms.

### Activity 3 · Remove a difference (10 minutes)

Run the box to see ten English consonants as patterns of **+** (marked) and **0** (the zero). Say each sound aloud and feel the difference: *p* vs *b*, *t* vs *d*.

Notice that **t** is zero on every feature. It is defined entirely by what it lacks.

Then put a feature name inside the brackets, for example `ignore=["voiced"]`, and run again. Which sounds can no longer be told apart? Try `["voiced", "nasal"]` too.

In [ ]:
sound_table(ignore=[])

Take away one difference and pairs of sounds collapse into one, as *pat* and *bat* would. Each sound was nothing but its pattern of differences.

## Part 4 · Letter and number: the mathematical parallel

Pourciau makes a bold side-claim: structuralist linguistics has more in common with the mathematics of its time than with the French literary theory it later inspired. She pairs **"letter and number, poetry and sets, rhyme and ordered pairs"**, but only sketches the comparison.

The shared problem was this. Between about 1870 and 1930, mathematicians could no longer treat numbers or the number line as simply given by intuition or by God. Like Saussure, they had to ground a system from inside, with nothing outside to lean on. Their answers make the same four moves:

| Move | In linguistics | In mathematics |
| --- | --- | --- |
| Things are defined only by relations | A sign is its difference from other signs | A number is a position in a sequence |
| Structure is built from emptiness | Jakobson's zero | The empty set |
| Order comes from repetition | Rhyme | The ordered pair |
| The system grows by a rule | Language as an ongoing event | Recursion; sets built in stages |

**Note for facilitators:** parts 4a and 4c follow comparisons Pourciau herself names. Parts 4b, 4d and 4e extend her sketch and were developed for this course.

### 4a · Dedekind's cut: a number defined as a gap *(Pourciau's own example)*

Fractions leave holes in the number line. No fraction multiplied by itself gives exactly 2. So what is √2?

In 1872 Richard Dedekind split all the fractions into two groups: a lower group A (whose squares are less than 2) and an upper group B (whose squares are more than 2). No fraction sits between them. Dedekind's move was to say the **gap itself is the number**. Nothing is hiding in it; the precisely defined split is √2.

Pourciau calls this **"negative writing"**: a number with no content of its own, fully defined by how it divides its neighbours. That is how Jakobson's zero works too.

### Activity 4 · Look for the number in the gap (10 minutes)

Run the box. Then raise `max_denominator` (try `50`, then `200`) to check more and more fractions. The two groups squeeze closer. Do they ever meet?

Then change `target` to `4`, then `9`. What is different?

In [ ]:
dedekind_cut(target=2, max_denominator=12)

For 4 and 9 a fraction sits at the split (2 and 3), so no new number is needed. For 2 there is only ever the gap. Two further points matter for Pourciau:

- **The gap depends on what surrounds it.** The cut exists only relative to the fractions it divides. In the same way, the structuralist system depends on the history it removes.
- **There is no shortcut.** You cannot start with the finished number line; you reach it only by passing through counting numbers, then fractions, then cuts.

### 4b · Numbers built from nothing *(extension)*

In 1888 Dedekind went further. He argued that numbers are simply whatever fills the positions in an endless sequence: a first element, a "next" rule, and no loops. What they are *in themselves* doesn't matter. That is Saussure's knight and button again.

But the sequence needs somewhere to start. The German logician Gottlob Frege (1884) defined **zero as the number of things that are not identical to themselves**. Nothing qualifies, so zero counts an absence. **One** is the number of things identical to zero: the first thing counted is the emptiness just established.

In 1923 John von Neumann turned this into the standard recipe, using **∅**, the empty collection. Each number is the collection of all the numbers before it:

- 0 = ∅
- 1 = {∅}
- 2 = {∅, {∅}}
- and so on.

The whole universe of modern mathematics rests on this single emptiness, gathered up again and again. It is the closest mathematical echo of Jakobson's zero: the foundation of the system is an ordinary written term whose content is nothing.

### Activity 5 · Build the numbers (5 minutes)

Run the box, then change `up_to` to `5` or `6`. Watch how fast the writing grows.

In [ ]:
nums = numbers_from_nothing(up_to=4)

**One more twist.** Other mathematicians built the numbers from ∅ in different ways. In 1965 the philosopher Paul Benacerraf asked which version is *really* the number 2, and answered: no fact decides it. Numbers are places in a structure, not objects. Keep this in mind for Part 5.

### 4c · Rhyme makes order *(Pourciau's pairing, developed here)*

A plain collection has no order: {a, b} is the same as {b, a}. But mathematics needs order: first and second, before and after. In 1921 Kazimierz Kuratowski produced order from unordered collections alone:

**(a, b) = { {a}, {a, b} }**

The trick is **repetition**. The item *a* appears twice and *b* once, and that imbalance is the only thing that puts *a* first.

Jakobson's 1960 essay "Linguistics and Poetics" says something very close about poetry. Poetry takes sameness (repeated sounds and rhythms) and uses it to organise the sequence of speech. Rhyme tells you where lines end and what belongs together. In both cases, a repetition that means nothing in itself creates order.

### Activity 6 · Order from repetition (10 minutes)

First, run the pair test. Swap in your own words for `"drum"` and `"voice"`.

In [ ]:
compare("drum", "voice")

Now let the rhymes reveal the structure of a poem. The program only looks at the last letters of each line; it knows nothing about meaning. Replace the verse with one of your own, or with lyrics you are allowed to share, and run it again. (Spelling is a rough guide to sound, so `letters=3` or `letters=1` may work better for some poems.)

In [ ]:
poem = """
The gallery is quiet after nine,
the last guest lingers by the door,
a spotlight hums along a single line,
and paint remembers what it's for.
"""
rhyme_scheme(poem, letters=2)

### 4d · Hilbert's meaningless signs *(extension)*

In 1899 David Hilbert rebuilt geometry from a list of axioms, defining points and lines only by how they relate. He reportedly said they could just as well be called tables, chairs and beer mugs. Frege objected that a definition must say what a thing *is*. Hilbert replied that if the axioms do not contradict each other, their objects exist.

This is the same argument Saussure was having: does a sign's value come from what it points to, or from its place in a system?

Hilbert later treated mathematical proofs as strings of marks moved around by rules, with meaning set aside. Saussure's private "anagram studies" did something similar with ancient verse, counting letters regardless of sense. Both made their systems rigorous by writing them down emptied of content. This is one way to read Pourciau's title, *The Writing of Spirit*.

### 4e · A rule, not a list *(Pourciau's link, developed here)*

A language is infinite but bounded. You can always make a new sentence, but not every string of words is one. A domain like that can't be written out as a list. It can only be defined by a **rule that applies to its own results**, which mathematicians call **recursion**.

### Activity 7 · One rule, endless sentences (5 minutes)

The rule below says: a *person* can be followed by "who", an action, and another *person*, and that person can be followed by "who"... and so on. Run it, then raise `depth`. Change the names in the Setup lists if you want to make it your own.

In [ ]:
sentences(depth=2, how_many=5)

Mathematics had its own version of this. Early set theory collapsed into paradoxes by assuming you could take "all sets" as one finished whole. The repair was to **build sets in stages**: start from ∅, and at each stage form every collection of what already exists. Run the box to see how the stages grow.

In [ ]:
sets_in_stages(stages=7)

There is no final stage. This matches Pourciau's picture of language as an **ongoing event**: governed by rules, not just habit, yet never a finished object. The paradoxes came from treating the whole as given all at once, which is roughly the nineteenth century's mistake about language spirit.

## Part 5 · Benacerraf in the machine

Here the history arrives at today's AI. A language model stores each word as a point in a space with hundreds of dimensions, called an **embedding**. A point means nothing on its own. Only distances and directions between points carry sense. Session 2 explores this with a real model; here is a toy version with two dimensions.

### Activity 8 · Turn the map (5 minutes)

Run the box. Then change the angle (try `45`, `180`) and try `mirror=True`. Every coordinate changes. Does anything that matters change?

In [ ]:
rotate_map(angle=90, mirror=False)

No arrangement is the "real" one, just as no collection is "really" the number 2. That is Saussure's knight, Dedekind's positions and Benacerraf's argument, now working inside every large language model. The line from structuralism runs through information theory (Claude Shannon's *bit* is a pure yes/no difference) and through the 1950s idea that a word's meaning shows in the company it keeps, to the embeddings AI is built on.

**Back to the big idea.** "It's just statistics" and "only humans mean things" are not the only options. Structuralism offers a third: meaning as a real, self-organising system that is neither a mind nor a list of frequencies. That is the territory Weatherby's *Language Machines* explores, and where Session 1 begins.

## Where the analogy strains

The mathematics shows how a system *could* ground itself coherently. It does not show how such a system arises among people. Three limits are worth naming:

- **Time is only a figure of speech in mathematics.** The "stages" of set-building are logical, not historical. Pourciau insists language's emptying happens in real use, over real time.
- **Mathematics has no speakers.** For Saussure and Jakobson, language forms where minds meet sounds. Mathematics has no such meeting point.
- **Gödel limits self-grounding.** In 1931 Kurt Gödel showed that a rich enough formal system cannot prove its own consistency from inside. That may mean Pourciau claims too much for structuralism. Or it may support her in a subtler way: Gödel's unprovable sentence is written *inside* the system and marks its edge, much as the zero does. The system cannot close its own ground, but it can write down where the ground is missing.

## Discuss

1. Look back at your answers to Activity 1. Would you move any statement into column C now?
2. Saussure's knight can be replaced by a button. Where in your own art form does the role matter more than the material?
3. Frege counts an absence to get zero. Where do silence, gaps or empty space carry structure in music, theatre, painting or dance?
4. If a model's words mean something only through their positions, is that meaning "real"? Does your answer change if you apply the same test to a dictionary?
5. Pourciau says the new system depends on the history it removes. Can you think of an artistic movement that defined itself by stripping something away?

## Glossary
- **Structuralism**: the tradition (Saussure, Jakobson) that treats meaning as a system of relationships.
- **Arbitrariness**: a sign's form is not tied to its meaning by nature; its value comes from contrast with other signs.
- **Sprachgeist**: "language spirit", the nineteenth-century idea of a living force driving a language through history.
- **Marked / unmarked**: a marked sound has a feature; an unmarked one is defined by lacking it.
- **Zero**: Jakobson's name for the unmarked, featureless side of a pair.
- **Empty set (∅)**: the collection with nothing in it.
- **Dedekind cut**: a split of the fractions into two groups; the split can define a new number.
- **Ordered pair**: two items where order matters, built from repetition.
- **Recursion**: a rule that applies to its own results, producing endless outputs from finite means.
- **Embedding**: a word stored as a point in a many-dimensional space, meaningful only by its position.

## Going further
- Sarah Pourciau, *The Writing of Spirit: Soul, System, and the Roots of Language Science* (Fordham University Press, 2017): the Introduction, Chapter 6 and the Afterword.
- Michel Foucault, *The Order of Things* (1966), especially the final chapter.
- Weatherby, *Language Machines*, Introduction and chapter 1.
- Roman Jakobson, "Linguistics and Poetics" (1960).
- Paul Benacerraf, "What Numbers Could Not Be", *Philosophical Review* (1965).
- Richard Dedekind, "Continuity and Irrational Numbers" (1872).